# Module 23: Week 12 BBO Optimization

## Last Exploration Week — Maximizing Final Gains

Week 11 delivered **5/8 new bests** (F2, F4, F5, F7, F8), the best single week of the entire project!

**This is the LAST exploration week.** Week 13 = EXACT_RETURN (submit best known points).

### Week 11 Results Summary
| Function | Week 11 Value | Previous Best | Result |
|----------|--------------|---------------|--------|
| F1 | 1.9156 | 1.993 (W10) | Regression (-3.9%, overshot from peak) |
| F2 | **0.7302** | 0.667 (W4) | **NEW BEST** (+9.5%, Noisy EI worked!) |
| F3 | -0.0316 | -0.0117 (W9) | Regression (overshot again) |
| F4 | **0.6293** | 0.6287 (W8) | **NEW BEST** (marginal, ultra-tight) |
| F5 | **1678.29** | 1675.3 (W10) | **NEW BEST** (+3.0, x0/x1 fine-tuning) |
| F6 | -0.6856 | -0.586 (W8) | Regression (3rd consecutive) |
| F7 | **2.5352** | 2.480 (W10) | **NEW BEST** (4th consecutive!) |
| F8 | **9.9389** | 9.937 (W10) | **NEW BEST** (4th consecutive!) |

### Week 12 Strategy: Conservative Exploitation with Proven Directions
- **F1**: Ultra-tight r=0.002 around W10 best. W11 overshot.
- **F2**: Noisy EI r=0.010 around W11 NEW best. Noisy function — any improvement is lucky.
- **F3**: Ultra-tight r=0.002 around W9 best. 3 overshoots confirm peak is there.
- **F4**: Ultra-tight r=0.002 around W11 NEW best. Extremely peaked.
- **F5**: Fine-tune x0/x1 r=0.003, pin x2/x3=1.0 around W11 best.
- **F6**: Ultra-tight r=0.003 around W8 best. Abandon all exploration.
- **F7**: Directional exploit r=0.008. 4 consecutive improvements! x0↓, x3↑.
- **F8**: Directional exploit r=0.005. 4 consecutive improvements! Continue trend.

In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import LeaveOneOut
from scipy.optimize import minimize
from scipy.stats import norm, qmc
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
sys.path.append(os.path.join(os.path.abspath('.'), '..', 'src'))

from utils import load_data, save_submission

np.random.seed(42)
print('Imports complete.')

Imports complete.


## Core BO Infrastructure (Output Warping + Multi-Kernel Ensemble + Multi-Acquisition)

In [2]:
def fit_gp_ensemble(X, y, dim, nu_list=[0.5, 1.5, 2.5], use_warping=True):
    """Fit multi-kernel GP ensemble with output warping."""
    # Output warping (Yeo-Johnson)
    if use_warping and len(y) >= 5:
        pt = PowerTransformer(method='yeo-johnson')
        y_warped = pt.fit_transform(y.reshape(-1, 1)).ravel()
    else:
        pt = None
        y_warped = y.copy()
    
    gps = []
    lml_scores = []
    loo_scores = []
    
    for nu in nu_list:
        kernel = ConstantKernel(1.0, (1e-4, 1e4)) * Matern(length_scale=np.ones(dim), nu=nu, length_scale_bounds=(1e-4, 1e2))
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=1e-6, normalize_y=True)
        try:
            gp.fit(X, y_warped)
            lml = gp.log_marginal_likelihood_value_
        except Exception:
            lml = -1e10
        
        # LOO-CV
        try:
            gp_loo = GaussianProcessRegressor(kernel=gp.kernel_, n_restarts_optimizer=0, alpha=1e-6, normalize_y=True)
            loo = LeaveOneOut()
            loo_errors = []
            for train_idx, test_idx in loo.split(X):
                gp_loo.fit(X[train_idx], y_warped[train_idx])
                pred = gp_loo.predict(X[test_idx])
                loo_errors.append((pred[0] - y_warped[test_idx[0]])**2)
            loo_score = -np.mean(loo_errors)
        except Exception:
            loo_score = -1e10
        
        gps.append(gp)
        lml_scores.append(lml)
        loo_scores.append(loo_score)
    
    # Compute weights: 60% LML + 40% LOO
    lml_arr = np.array(lml_scores)
    loo_arr = np.array(loo_scores)
    
    def softmax(x):
        x = x - np.max(x)
        e = np.exp(x)
        return e / e.sum()
    
    lml_w = softmax(lml_arr)
    loo_w = softmax(loo_arr)
    weights = 0.6 * lml_w + 0.4 * loo_w
    weights /= weights.sum()
    
    print(f"  Kernel weights: {dict(zip([f'nu={nu}' for nu in nu_list], [f'{w:.3f}' for w in weights]))}")
    print(f"  LML scores: {dict(zip([f'nu={nu}' for nu in nu_list], [f'{s:.1f}' for s in lml_scores]))}")
    
    return gps, weights, pt


def ensemble_predict(gps, weights, X_test):
    """Weighted ensemble prediction."""
    mu_total = np.zeros(len(X_test))
    var_total = np.zeros(len(X_test))
    
    for gp, w in zip(gps, weights):
        mu, sigma = gp.predict(X_test, return_std=True)
        mu_total += w * mu
        var_total += w * (sigma**2 + mu**2)
    
    var_total -= mu_total**2
    sigma_total = np.sqrt(np.maximum(var_total, 1e-10))
    
    return mu_total, sigma_total


def multi_acquisition(mu, sigma, y_best, pi_weight=0.5, ei_weight=0.3, ucb_weight=0.2, kappa=1.96):
    """Multi-acquisition ensemble: normalized PI + EI + UCB."""
    z = (mu - y_best) / (sigma + 1e-10)
    
    # PI
    pi = norm.cdf(z)
    
    # EI 
    ei = (mu - y_best) * norm.cdf(z) + sigma * norm.pdf(z)
    ei = np.maximum(ei, 0)
    
    # UCB
    ucb = mu + kappa * sigma
    
    # Normalize each to [0, 1]
    def normalize(a):
        r = a.max() - a.min()
        if r < 1e-10:
            return np.ones_like(a) * 0.5
        return (a - a.min()) / r
    
    score = pi_weight * normalize(pi) + ei_weight * normalize(ei) + ucb_weight * normalize(ucb)
    return score


def noisy_ei(mu, sigma, gp_predicted_best):
    """Noisy EI using GP-predicted incumbent (for noisy functions)."""
    z = (mu - gp_predicted_best) / (sigma + 1e-10)
    ei = (mu - gp_predicted_best) * norm.cdf(z) + sigma * norm.pdf(z)
    return np.maximum(ei, 0)


print('Core functions defined.')

Core functions defined.


## Trust Region Optimization Engine

In [3]:
from scipy.stats.qmc import Sobol

def trust_region_optimize(func_id, dim, center, radius, 
                          acq_type='multi', pi_weight=0.5, ei_weight=0.3, ucb_weight=0.2,
                          constraints=None, n_candidates=20000, use_warping=True,
                          directional_bias=None):
    """Full trust-region BO pipeline."""
    print(f"\n{'='*60}")
    print(f"Function {func_id} (dim={dim})")
    print(f"Center: {center}")
    print(f"Radius: {radius}, Acq: {acq_type}")
    print(f"{'='*60}")
    
    # Load data
    df = load_data(func_id)
    X = df.iloc[:, :dim].values
    y = df['y'].values
    
    y_best_observed = y.max()
    best_idx = y.argmax()
    x_best_observed = X[best_idx]
    print(f"  Best observed: y={y_best_observed:.6f} at {x_best_observed}")
    print(f"  Total samples: {len(y)}")
    
    # Fit GP ensemble
    gps, weights, pt = fit_gp_ensemble(X, y, dim, use_warping=use_warping)
    
    # Generate Sobol candidates in trust region
    sobol = Sobol(d=dim, scramble=True, seed=42)
    sobol_raw = sobol.random(n_candidates)
    
    # Scale to trust region
    center_arr = np.array(center)
    if isinstance(radius, (list, np.ndarray)):
        radius_arr = np.array(radius)
    else:
        radius_arr = np.ones(dim) * radius
    
    candidates = center_arr - radius_arr + 2 * radius_arr * sobol_raw
    
    # Apply directional bias if provided
    if directional_bias is not None:
        # directional_bias is a dict: {dim_idx: (direction, strength)}
        # direction: +1 or -1, strength: fraction of candidates to bias
        for d_idx, (direction, strength) in directional_bias.items():
            n_biased = int(n_candidates * strength)
            if direction > 0:
                # Bias upper half
                candidates[:n_biased, d_idx] = center_arr[d_idx] + radius_arr[d_idx] * np.abs(sobol_raw[:n_biased, d_idx])
            else:
                # Bias lower half
                candidates[:n_biased, d_idx] = center_arr[d_idx] - radius_arr[d_idx] * np.abs(sobol_raw[:n_biased, d_idx])
    
    # Clip to [0, 1]
    candidates = np.clip(candidates, 0.0, 1.0)
    
    # Apply constraints
    if constraints:
        for d_idx, (lo, hi) in constraints.items():
            candidates[:, d_idx] = np.clip(candidates[:, d_idx], lo, hi)
    
    # Predict with ensemble
    mu, sigma = ensemble_predict(gps, weights, candidates)
    
    # Inverse-warp predictions for acquisition if warping was used
    if pt is not None:
        # y_best in warped space for acquisition
        y_best_warped = pt.transform(np.array([[y_best_observed]])).ravel()[0]
        # Also get GP-predicted incumbent in warped space
        mu_train, _ = ensemble_predict(gps, weights, X)
        gp_predicted_best_warped = mu_train.max()
    else:
        y_best_warped = y_best_observed
        mu_train, _ = ensemble_predict(gps, weights, X)
        gp_predicted_best_warped = mu_train.max()
    
    # Compute acquisition
    if acq_type == 'multi':
        scores = multi_acquisition(mu, sigma, y_best_warped, pi_weight, ei_weight, ucb_weight)
    elif acq_type == 'noisy_ei':
        scores = noisy_ei(mu, sigma, gp_predicted_best_warped)
    elif acq_type == 'pi':
        z = (mu - y_best_warped) / (sigma + 1e-10)
        scores = norm.cdf(z)
    else:
        scores = multi_acquisition(mu, sigma, y_best_warped)
    
    # Select best
    best_cand_idx = np.argmax(scores)
    x_next = candidates[best_cand_idx]
    mu_next = mu[best_cand_idx]
    sigma_next = sigma[best_cand_idx]
    
    # Inverse warp the predicted value for display
    if pt is not None:
        mu_display = pt.inverse_transform(np.array([[mu_next]])).ravel()[0]
    else:
        mu_display = mu_next
    
    print(f"\n  Selected: {x_next}")
    print(f"  GP predicted (original scale): {mu_display:.6f} ± {sigma_next:.6f}")
    print(f"  Distance from center: {np.linalg.norm(x_next - center_arr):.6f}")
    print(f"  Distance from best observed: {np.linalg.norm(x_next - x_best_observed):.6f}")
    
    return x_next, mu_display, sigma_next

print('Trust region optimizer defined.')

Trust region optimizer defined.


## Function 1: Ultra-tight around W10 best [0.6297, 0.6287]

W11 submitted [0.6317, 0.6314] → 1.916 (regression). The peak is extremely narrow.
W10 best [0.6297, 0.6287] → 1.993 remains the best.

Strategy: r=0.002, high PI weight (0.6) for pure exploitation.

In [4]:
x_next_f1, mu_f1, sigma_f1 = trust_region_optimize(
    func_id=1, dim=2,
    center=[0.629731, 0.628741],  # W10 best (1.993)
    radius=0.002,
    acq_type='multi', pi_weight=0.6, ei_weight=0.2, ucb_weight=0.2,
    use_warping=True
)
print(f"\nF1 query: {'-'.join(f'{v:.6f}' for v in x_next_f1)}")


Function 1 (dim=2)
Center: [0.629731, 0.628741]
Radius: 0.002, Acq: multi
  Best observed: y=1.993006 at [0.629731 0.628741]
  Total samples: 21


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  Kernel weights: {'nu=0.5': '0.144', 'nu=1.5': '0.270', 'nu=2.5': '0.586'}
  LML scores: {'nu=0.5': '-15.6', 'nu=1.5': '-11.3', 'nu=2.5': '-10.1'}

  Selected: [0.62977178 0.6271747 ]
  GP predicted (original scale): 1.973911 ± 0.048273
  Distance from center: 0.001567
  Distance from best observed: 0.001567

F1 query: 0.629772-0.627175


/var/folders/dl/z8gs0zfd15zggds6w580lbtr0000gr/T/ipykernel_49406/946587184.py:30: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sobol_raw = sobol.random(n_candidates)


## Function 2: Noisy EI around W11 NEW BEST [0.7026, 0.9189]

W11 found NEW BEST 0.7302 at [0.7026, 0.9189]! Noisy EI with x1<0.920 constraint worked.
F2 is definitively noisy (same point gave 0.611, 0.667, 0.590). Use Noisy EI.

Strategy: Noisy EI, r=0.010, explore around the new best region.

In [5]:
x_next_f2, mu_f2, sigma_f2 = trust_region_optimize(
    func_id=2, dim=2,
    center=[0.702597, 0.918887],  # W11 NEW best (0.7302)
    radius=0.010,
    acq_type='noisy_ei',
    use_warping=True
)
print(f"\nF2 query: {'-'.join(f'{v:.6f}' for v in x_next_f2)}")


Function 2 (dim=2)
Center: [0.702597, 0.918887]
Radius: 0.01, Acq: noisy_ei
  Best observed: y=0.730244 at [0.702597 0.918887]
  Total samples: 21


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 5 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lo

  Kernel weights: {'nu=0.5': '0.742', 'nu=1.5': '0.129', 'nu=2.5': '0.129'}
  LML scores: {'nu=0.5': '-41661.5', 'nu=1.5': '-41680.0', 'nu=2.5': '-41685.0'}

  Selected: [0.70270231 0.9265839 ]
  GP predicted (original scale): 0.750254 ± 2.338994
  Distance from center: 0.007698
  Distance from best observed: 0.007698

F2 query: 0.702702-0.926584


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decre

## Function 3: Ultra-tight around W9 best [0.5296, 0.6390, 0.3896]

W11 submitted [0.5327, 0.6440, 0.3879] → -0.0316 (regression). W10 also regressed.
Every attempt to move from W9 has regressed. Peak is extremely close to W9 position.

Strategy: r=0.002, high PI weight (0.7) for maximum exploitation.

In [6]:
x_next_f3, mu_f3, sigma_f3 = trust_region_optimize(
    func_id=3, dim=3,
    center=[0.529607, 0.638966, 0.389611],  # W9 best (-0.0117)
    radius=0.002,
    acq_type='multi', pi_weight=0.7, ei_weight=0.15, ucb_weight=0.15,
    use_warping=True
)
print(f"\nF3 query: {'-'.join(f'{v:.6f}' for v in x_next_f3)}")

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decre


Function 3 (dim=3)
Center: [0.529607, 0.638966, 0.389611]
Radius: 0.002, Acq: multi
  Best observed: y=-0.011662 at [0.529607 0.638966 0.389611]
  Total samples: 26


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decre

  Kernel weights: {'nu=0.5': '0.733', 'nu=1.5': '0.133', 'nu=2.5': '0.133'}
  LML scores: {'nu=0.5': '-35.2', 'nu=1.5': '-59.6', 'nu=2.5': '-66.2'}

  Selected: [0.52967053 0.63899913 0.38953014]
  GP predicted (original scale): -0.043233 ± 1.724814
  Distance from center: 0.000108
  Distance from best observed: 0.000108

F3 query: 0.529671-0.638999-0.389530


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 3 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified lo

## Function 4: Ultra-tight around W11 NEW BEST [0.4204, 0.3809, 0.4102, 0.4219]

W11 found NEW BEST 0.6293 at [0.4204, 0.3809, 0.4102, 0.4219] (marginal improvement over W8's 0.6287).
This function is extremely peaked — r=0.010 caused 15% regression (W10).

Strategy: r=0.002, PI-heavy (0.7) for maximum exploitation.

In [7]:
x_next_f4, mu_f4, sigma_f4 = trust_region_optimize(
    func_id=4, dim=4,
    center=[0.420406, 0.380892, 0.410213, 0.421927],  # W11 NEW best (0.6293)
    radius=0.002,
    acq_type='multi', pi_weight=0.7, ei_weight=0.15, ucb_weight=0.15,
    use_warping=True
)
print(f"\nF4 query: {'-'.join(f'{v:.6f}' for v in x_next_f4)}")


Function 4 (dim=4)
Center: [0.420406, 0.380892, 0.410213, 0.421927]
Radius: 0.002, Acq: multi
  Best observed: y=0.629336 at [0.420406 0.380892 0.410213 0.421927]
  Total samples: 41


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 12 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


  Kernel weights: {'nu=0.5': '0.568', 'nu=1.5': '0.313', 'nu=2.5': '0.119'}
  LML scores: {'nu=0.5': '-7.1', 'nu=1.5': '-7.9', 'nu=2.5': '-15.1'}

  Selected: [0.42086094 0.38045967 0.41117577 0.42387757]
  GP predicted (original scale): 0.632629 ± 0.025527
  Distance from center: 0.002264
  Distance from best observed: 0.002264

F4 query: 0.420861-0.380460-0.411176-0.423878


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/var/folders/dl/z8gs0zfd15zggds6w580lbtr0000gr/T/ipykernel_49406/946587184.py:30: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sobol_raw = sobol.random(n_candidates)


## Function 5: Fine-tune x0/x1 around W11 best, pin x2/x3=1.0

W11 found NEW BEST 1678.29 at [0.3730, 0.2810, 1.0, 1.0].
x2/x3 confirmed at 1.0. Only x0/x1 need fine-tuning.

Strategy: r=0.003 for x0/x1, pin x2/x3 at [0.999999, 1.0].

In [8]:
x_next_f5, mu_f5, sigma_f5 = trust_region_optimize(
    func_id=5, dim=4,
    center=[0.372961, 0.280952, 0.999999, 0.999999],  # W11 NEW best (1678.29)
    radius=[0.003, 0.003, 0.000001, 0.000001],  # Only move x0/x1
    acq_type='multi', pi_weight=0.5, ei_weight=0.3, ucb_weight=0.2,
    constraints={2: (0.999998, 1.0), 3: (0.999998, 1.0)},
    use_warping=True
)
print(f"\nF5 query: {'-'.join(f'{v:.6f}' for v in x_next_f5)}")


Function 5 (dim=4)
Center: [0.372961, 0.280952, 0.999999, 0.999999]
Radius: [0.003, 0.003, 1e-06, 1e-06], Acq: multi
  Best observed: y=1678.291903 at [0.372961 0.280952 0.999999 0.999999]
  Total samples: 31


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

  Kernel weights: {'nu=0.5': '0.125', 'nu=1.5': '0.165', 'nu=2.5': '0.710'}
  LML scores: {'nu=0.5': '1.6', 'nu=1.5': '18.4', 'nu=2.5': '21.4'}

  Selected: [0.37593212 0.28391582 0.9999985  0.99999976]
  GP predicted (original scale): 1680.323242 ± 0.002936
  Distance from center: 0.004197
  Distance from best observed: 0.004197

F5 query: 0.375932-0.283916-0.999998-1.000000


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 10 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 13 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/var/folders/dl/z8gs0zfd15zggds6w580lbtr0000gr/T/ipykernel_49406/946587184.py:30: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sobol_raw = sobol.random(n_candidates)


## Function 6: Ultra-tight around W8 best [0.6902, 0.1258, 0.7578, 0.7367, 0.0510]

3 consecutive regressions from W8 best (W9, W10, W11). W11 at [0.6900, 0.1257, 0.7588, 0.7377, 0.0507] 
was very close to W8 and still regressed (-0.686 vs -0.586). The peak is extremely sharp.

Strategy: r=0.003, PI-heavy (0.7), stay as close to W8 best as possible.

In [9]:
x_next_f6, mu_f6, sigma_f6 = trust_region_optimize(
    func_id=6, dim=5,
    center=[0.690234, 0.125757, 0.757804, 0.736682, 0.050971],  # W8 best (-0.586)
    radius=0.003,
    acq_type='multi', pi_weight=0.7, ei_weight=0.15, ucb_weight=0.15,
    use_warping=True
)
print(f"\nF6 query: {'-'.join(f'{v:.6f}' for v in x_next_f6)}")


Function 6 (dim=5)
Center: [0.690234, 0.125757, 0.757804, 0.736682, 0.050971]
Radius: 0.003, Acq: multi
  Best observed: y=-0.586391 at [0.690234 0.125757 0.757804 0.736682 0.050971]
  Total samples: 31


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

  Kernel weights: {'nu=0.5': '0.787', 'nu=1.5': '0.107', 'nu=2.5': '0.106'}
  LML scores: {'nu=0.5': '-27.5', 'nu=1.5': '-38.2', 'nu=2.5': '-38.7'}

  Selected: [0.69009579 0.12817106 0.75590136 0.73663874 0.05089048]
  GP predicted (original scale): -0.679452 ± 0.791211
  Distance from center: 0.003078
  Distance from best observed: 0.003078

F6 query: 0.690096-0.128171-0.755901-0.736639-0.050890


## Function 7: Directional exploit — 4 consecutive improvements!

Progress: W8→W9→W10→W11: 2.433→2.448→2.480→2.535 (4 consecutive improvements!).

Key trends from W8→W11:
- x0: 0.010→0.014→0.014→0.004 (trending down, W11 big decrease worked!)
- x1: 0.108→0.131→0.134→0.141 (slowly increasing)
- x2: 0.581→0.580→0.583→0.574 (relatively stable)
- x3: 0.206→0.217→0.233→0.243 (consistently increasing)
- x4: 0.365→0.371→0.371→0.364 (relatively stable)
- x5: 0.740→0.748→0.745→0.736 (relatively stable)

Strategy: Continue directional trend, r=0.008. Bias x0↓, x3↑.

In [10]:
x_next_f7, mu_f7, sigma_f7 = trust_region_optimize(
    func_id=7, dim=6,
    center=[0.004063, 0.141375, 0.574198, 0.242683, 0.363648, 0.735645],  # W11 NEW best (2.535)
    radius=0.008,
    acq_type='multi', pi_weight=0.5, ei_weight=0.3, ucb_weight=0.2,
    constraints={0: (0.0, 0.012)},  # Keep x0 low
    directional_bias={0: (-1, 0.7), 3: (+1, 0.6)},  # x0 down, x3 up
    use_warping=True
)
print(f"\nF7 query: {'-'.join(f'{v:.6f}' for v in x_next_f7)}")


Function 7 (dim=6)
Center: [0.004063, 0.141375, 0.574198, 0.242683, 0.363648, 0.735645]
Radius: 0.008, Acq: multi
  Best observed: y=2.535207 at [0.004063 0.141375 0.574198 0.242683 0.363648 0.735645]
  Total samples: 41


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

  Kernel weights: {'nu=0.5': '0.131', 'nu=1.5': '0.178', 'nu=2.5': '0.691'}
  LML scores: {'nu=0.5': '-19.7', 'nu=1.5': '-4.2', 'nu=2.5': '-1.7'}

  Selected: [0.         0.13979885 0.57192111 0.2492734  0.3559908  0.72779025]
  GP predicted (original scale): 2.543100 ± 0.049069
  Distance from center: 0.013709
  Distance from best observed: 0.013709

F7 query: 0.000000-0.139799-0.571921-0.249273-0.355991-0.727790


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

## Function 8: Directional exploit — 4 consecutive improvements!

Progress: W8→W9→W10→W11: 9.928→9.933→9.937→9.939 (4 consecutive improvements!).

Key trends from W8→W11:
- x0: 0.026→0.033→0.034→0.035 (slowly increasing)
- x1: 0.095→0.087→0.087→0.082 (decreasing)
- x2: 0.153→0.149→0.148→0.143 (decreasing)
- x3: 0.049→0.058→0.059→0.062 (increasing)
- x4: 0.871→0.881→0.878→0.874 (relatively stable)
- x5: 0.334→0.343→0.350→0.356 (increasing)
- x6: 0.170→0.166→0.167→0.161 (decreasing slightly)
- x7: 0.223→0.227→0.235→0.237 (increasing)

Strategy: Continue directional trend, r=0.005. Keep x0<0.040.

In [11]:
x_next_f8, mu_f8, sigma_f8 = trust_region_optimize(
    func_id=8, dim=8,
    center=[0.034514, 0.081578, 0.142709, 0.061672, 0.873622, 0.356217, 0.160895, 0.237298],  # W11 NEW best (9.939)
    radius=0.005,
    acq_type='multi', pi_weight=0.5, ei_weight=0.3, ucb_weight=0.2,
    constraints={0: (0.0, 0.040)},  # Keep x0 low
    directional_bias={1: (-1, 0.6), 2: (-1, 0.6), 5: (+1, 0.6), 7: (+1, 0.6)},
    use_warping=True
)
print(f"\nF8 query: {'-'.join(f'{v:.6f}' for v in x_next_f8)}")


Function 8 (dim=8)
Center: [0.034514, 0.081578, 0.142709, 0.061672, 0.873622, 0.356217, 0.160895, 0.237298]
Radius: 0.005, Acq: multi
  Best observed: y=9.938858 at [0.034514 0.081578 0.142709 0.061672 0.873622 0.356217 0.160895 0.237298]
  Total samples: 51


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 74 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 38 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is 

/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

  Kernel weights: {'nu=0.5': '0.130', 'nu=1.5': '0.136', 'nu=2.5': '0.733'}
  LML scores: {'nu=0.5': '6.9', 'nu=1.5': '46.8', 'nu=2.5': '52.7'}

  Selected: [0.03881751 0.08156308 0.1384006  0.06659949 0.86898605 0.36097161
 0.16342779 0.24042264]
  GP predicted (original scale): 9.942233 ± 0.020308
  Distance from center: 0.011029
  Distance from best observed: 0.011029

F8 query: 0.038818-0.081563-0.138401-0.066599-0.868986-0.360972-0.163428-0.240423


/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/benjaminbaumann/Documents/Projects/capstone-project-icl/.venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasi

## Summary & Submissions

In [12]:
results = {
    1: x_next_f1,
    2: x_next_f2,
    3: x_next_f3,
    4: x_next_f4,
    5: x_next_f5,
    6: x_next_f6,
    7: x_next_f7,
    8: x_next_f8
}

# Current best values
current_bests = {
    1: ('W10', 1.993, [0.629731, 0.628741]),
    2: ('W11', 0.7302, [0.702597, 0.918887]),
    3: ('W9', -0.0117, [0.529607, 0.638966, 0.389611]),
    4: ('W11', 0.6293, [0.420406, 0.380892, 0.410213, 0.421927]),
    5: ('W11', 1678.29, [0.372961, 0.280952, 0.999999, 0.999999]),
    6: ('W8', -0.586, [0.690234, 0.125757, 0.757804, 0.736682, 0.050971]),
    7: ('W11', 2.535, [0.004063, 0.141375, 0.574198, 0.242683, 0.363648, 0.735645]),
    8: ('W11', 9.939, [0.034514, 0.081578, 0.142709, 0.061672, 0.873622, 0.356217, 0.160895, 0.237298])
}

print('=' * 80)
print('WEEK 12 SUBMISSION SUMMARY (LAST EXPLORATION WEEK)')
print('=' * 80)
print(f'{"Func":>4} | {"Current Best":>14} | {"Best Week":>10} | {"W12 Query":>50} | {"Dist from Best":>14}')
print('-' * 110)

for fid in range(1, 9):
    x = results[fid]
    week, best_val, best_loc = current_bests[fid]
    dist = np.linalg.norm(x - np.array(best_loc))
    query_str = '-'.join(f'{v:.6f}' for v in x)
    print(f'  F{fid} | {best_val:>14.6f} | {week:>10} | {query_str:>50} | {dist:>14.6f}')

print('\n')
print('SUBMISSION STRINGS (copy-paste ready):')
print('-' * 60)
for fid in range(1, 9):
    x = results[fid]
    query_str = '-'.join(f'{v:.6f}' for v in x)
    print(f'Function {fid}: {query_str}')

WEEK 12 SUBMISSION SUMMARY (LAST EXPLORATION WEEK)
Func |   Current Best |  Best Week |                                          W12 Query | Dist from Best
--------------------------------------------------------------------------------------------------------------
  F1 |       1.993000 |        W10 |                                  0.629772-0.627175 |       0.001567
  F2 |       0.730200 |        W11 |                                  0.702702-0.926584 |       0.007698
  F3 |      -0.011700 |         W9 |                         0.529671-0.638999-0.389530 |       0.000108
  F4 |       0.629300 |        W11 |                0.420861-0.380460-0.411176-0.423878 |       0.002264
  F5 |    1678.290000 |        W11 |                0.375932-0.283916-0.999998-1.000000 |       0.004197
  F6 |      -0.586000 |         W8 |       0.690096-0.128171-0.755901-0.736639-0.050890 |       0.003078
  F7 |       2.535000 |        W11 | 0.000000-0.139799-0.571921-0.249273-0.355991-0.727790 |       0.01

In [13]:
# Save submissions
for fid in range(1, 9):
    x = results[fid]
    query_str = '-'.join(f'{v:.6f}' for v in x)
    save_submission(fid, query_str, module_name='Module 23')
    
print('\nAll submissions saved!')

Saved submission for Function 1 to submissions/submission_log.csv
Saved submission for Function 2 to submissions/submission_log.csv
Saved submission for Function 3 to submissions/submission_log.csv
Saved submission for Function 4 to submissions/submission_log.csv
Saved submission for Function 5 to submissions/submission_log.csv
Saved submission for Function 6 to submissions/submission_log.csv
Saved submission for Function 7 to submissions/submission_log.csv
Saved submission for Function 8 to submissions/submission_log.csv

All submissions saved!


## Week 13 Preview: EXACT_RETURN

Next week is the FINAL week. We submit the BEST KNOWN point for each function:
- F1: [0.629731, 0.628741] → 1.993 (W10)
- F2: Best from W11 or W12 (noisy — best luck)
- F3: [0.529607, 0.638966, 0.389611] → -0.0117 (W9)
- F4: Best from W8/W11/W12 
- F5: Best from W11/W12
- F6: [0.690234, 0.125757, 0.757804, 0.736682, 0.050971] → -0.586 (W8)
- F7: Best from W11/W12 (if W12 improves)
- F8: Best from W11/W12 (if W12 improves)

## Part 2: Reflection on Optimisation Strategy

### 1. How has your optimisation strategy evolved since your first few rounds of queries? Which elements now feel more structured or systematic?

Looking back, the early weeks feel almost naive compared to where we are now. At the start, the approach was quite reactive. We were fitting basic Gaussian Processes with a single kernel, using a standard acquisition function, and essentially hoping that the model would point us somewhere useful. There was a lot of trial and error, and when a query didn't work out, the main response was just to try something slightly different the next week without much rigour behind the adjustment.

Over time, the pipeline became genuinely systematic. The surrogate model evolved from a single GP into a multi-kernel ensemble that automatically selects the appropriate smoothness for each function. We added output warping to handle skewed distributions, trust regions to control how far we search from known good points, and a multi-acquisition ensemble that blends several scoring criteria rather than relying on a single one. Each of these additions was motivated by a specific failure or insight from earlier weeks.

But the biggest evolution has probably been in how we *think* about each function. We now classify functions by their character (peaked versus flat, noisy versus deterministic, interior versus boundary optimum) and apply a corresponding strategy template. That framework was not designed upfront. It emerged organically from spending weeks analysing why certain queries improved and others regressed. The whole process feels much more like engineering now and much less like guessing.

---

### 2. If you think of your current data set as a 'high-dimensional' space, which variables or behaviours seem to drive the largest variation in your results, similar to principal components in PCA?

If we imagine all the strategy choices we make each week (trust region radius, where to centre the search, which acquisition function to use, kernel smoothness, etc.) as dimensions in a high-dimensional space, then the "principal components" that explain the most variance in our outcomes become fairly clear.

The single most impactful factor is the **trust region radius**. For functions with narrow peaks, getting this wrong by even a small amount leads to immediate regression. It explains more of the outcome variance than any modelling decision. The second most influential factor is **where we centre the search and how we bias the direction**. For functions that showed consistent directional improvement, following that gradient was the key driver of consecutive gains. Conversely, choosing the wrong direction led to repeated regressions on other functions.

After those two, the **choice of acquisition function** matters noticeably, particularly the decision of whether to account for observation noise. This turned out to be the key unlock for one of our noisier functions. Further down the list, kernel smoothness still matters but is largely handled automatically by the ensemble, and the remaining pipeline details (output warping, candidate generation settings) contribute to robustness without strongly determining individual outcomes.

The parallel to PCA is quite direct: a handful of strategy decisions capture the vast majority of outcome variance, and everything else is a comparatively minor detail.

---

### 3. How do you decide which aspects of your strategy to keep exploring versus which to reduce or simplify, as PCA reduces dimensions while retaining essential information?

The logic is essentially the same as PCA's variance-retention rule: keep refining the aspects that explain the most outcome variance, and fix or automate the rest.

We still actively tune the trust region radius and the search centre each week, because these have the highest impact and they depend on how the landscape looks locally, which changes as we gather more data near the peak. These are the "high-variance components" that we cannot afford to simplify yet.

On the other hand, we have progressively locked down the parts of the pipeline that turned out not to matter much on a per-function basis. Kernel selection is now fully automated by the ensemble. Output warping is always on. Candidate generation uses a fixed Sobol sequence size. For one function where we confirmed two of four input dimensions should sit at the boundary, we effectively projected the problem from 4D down to 2D by pinning those dimensions permanently.

The PCA analogy feels quite apt here. Just as you would discard low-variance principal components to reduce dimensionality without losing meaningful information, we have "projected out" the strategy dimensions that do not meaningfully affect outcomes. That frees up attention and effort for the decisions that genuinely drive results.

---

### 4. How might this round of optimisation influence your next and final round of query submission in Module 24, especially when balancing exploration and exploitation?

The final week will be a pure exploitation round with no exploration at all. The plan is straightforward: for each function, compare the Week 12 result against the all-time best, and submit whichever point performed better.

The functions that have shown consistent directional improvement over the last several weeks are the ones most likely to benefit from Week 12's queries. If the trend holds, this week's point becomes the new submission target. For the functions that have very narrow peaks and where we used ultra-tight search radii, we do not expect large changes in either direction, so the existing best will likely stand.

The noisy function is the one genuine wildcard. Because repeated evaluations of the same input have returned different values, there is no guarantee that even the "best" observed point will reproduce. For the final week, we will simply submit the coordinates that yielded the highest observation, while accepting that noise may cause the actual result to vary.

The overarching lesson is that the final week should introduce no new risk. Every previous regression was caused by moving away from a known good point. Returning to proven locations eliminates that risk entirely, which feels like the right trade-off when there is nothing left to learn.

---

### 5. Reflect briefly on how insights from PCA, such as focusing on variance and removing redundancy, might apply to how you interpret your BBO results.

PCA's core idea of identifying the directions of maximum variance and discarding the rest maps quite naturally onto how we have come to interpret our BBO results.

**Focusing on variance.** Our optimisation experience revealed that a small number of strategy decisions (search radius, centre location) account for most of the spread in outcomes, while many other pipeline details add little. This mirrors how the first few principal components in PCA typically capture the bulk of total variance while the remaining components contribute diminishing returns.

**Removing redundancy.** We found that some strategy choices were effectively redundant. For example, separately tuning three acquisition function weights per function turned out to be unnecessary: a simple binary choice between "exploitation-heavy" and "balanced" weightings captured nearly all the useful variation. Similarly, once we confirmed that certain input dimensions should be fixed at the boundary, those dimensions became redundant and could be projected out entirely, collapsing the effective problem dimensionality.

**Interpreting the residual.** PCA teaches us that not all variation is signal; some is noise to be discarded. One of our functions demonstrated this vividly: repeated evaluations of the exact same input returned meaningfully different outputs. That variation was pure observation noise, not useful information about the landscape. Recognising this distinction and adapting the acquisition function accordingly was one of our most impactful methodological shifts.

**Dimensionality reduction as a strategy.** The overall arc of this project has been one of progressive dimensionality reduction, not of the input space per se, but of the strategy space. We started with many tuneable parameters and progressively fixed or automated the ones that did not matter, converging on a small set of decisions that actually drive outcomes. In a sense, that is PCA applied to the meta-problem of optimisation strategy design itself.